In [1]:
import os
import pandas as pd
import sys
sys.path.append(os.getcwd())
import utils

import joblib

print("✅ All libraries are working!")


✅ All libraries are working!


### Forecasting player's fantasy performance at 2025-26
Based on our modelling section we will now proceed with predicting the leadgue ranking toward the upcoming season

In [2]:
# import variables from last notebook
predict_only = pd.read_pickle("../data/predict_only.pkl")

feature_cols_df = pd.read_pickle("../data/feature_cols.pickle")
feature_cols= feature_cols_df[0].tolist()

df = pd.read_pickle("../data/df.pickle")
linear_train = pd.read_pickle("../data/linear_train.pickle")
train = pd.read_pickle("../data/train.pickle")
linear_test = pd.read_pickle("../data/linear_test.pickle")
test = pd.read_pickle("../data/test.pickle")

pipe_RF = joblib.load("pipe_RF.pkl")
preprocessor = joblib.load("preprocessor.pkl")
ols_model = joblib.load("ols_model.pkl")
final_features = joblib.load("final_features.pkl")

In [3]:
df_forecast = predict_only.copy()
forecasts = {}

In [4]:
# Random Forest predictions
X_forecast = predict_only[feature_cols]
forecast_RF = pipe_RF.predict(X_forecast)
forecasts["pred_RandomForest"] = forecast_RF
df_forecast["pred_RandomForest"] = forecast_RF

In [5]:
# OLS predictions
X_fore_arr = preprocessor.transform(predict_only)
X_fore_df = pd.DataFrame(X_fore_arr, columns=preprocessor.get_feature_names_out(), index=predict_only.index)

preds_OLS = utils.forecast_with_ols(ols_model, X_fore_df, final_features)

forecasts["OLS"] = preds_OLS
df_forecast = df_forecast.copy()
df_forecast.loc[:, "pred_OLS"] = preds_OLS

print("NaNs in predictions:", preds_OLS.isna().sum())

NaNs in predictions: 0


In [6]:
forecast_results = df_forecast[["PLAYER_NAME", "SEASON_ID", "fantasy_z_9cat", "pred_OLS", "pred_RandomForest"]]

# Rank top players by prediction
forecast_top = forecast_results.sort_values("pred_OLS", ascending=False).head(20)
print(forecast_top)

#df_forecast.to_csv("forecast_to_2025-26.csv", index = False)

                  PLAYER_NAME SEASON_ID  fantasy_z_9cat   pred_OLS  \
1921        Victor Wembanyama   2024-25       16.605037  16.719948   
1506             Nikola Jokić   2024-25       17.539191  15.624058   
1740  Shai Gilgeous-Alexander   2024-25       16.562903  14.886174   
87              Anthony Davis   2024-25       11.999300  12.661495   
692     Giannis Antetokounmpo   2024-25        7.142473  11.306681   
1312              Luka Dončić   2024-25       10.321897  11.207628   
1903        Tyrese Haliburton   2024-25       10.194490  10.673968   
1909             Tyrese Maxey   2024-25        9.427006   9.956577   
941              Jayson Tatum   2024-25        9.204196   9.810708   
988               Joel Embiid   2024-25        7.645750   9.179041   
898         Jaren Jackson Jr.   2024-25        7.665074   9.140564   
92            Anthony Edwards   2024-25        8.628321   8.915679   
1233       Kristaps Porziņģis   2024-25        8.101312   8.914970   
232           Cade C

**Applying best models on forecastig all categories - will help later to create a team-building reccomandation which will be based on punt strategies**

In [7]:
# define raw_feature_columns exactly as used when fitting preprocessor:
raw_feature_columns = feature_cols

train_df = df[df["SEASON_ID"] < "2024-25"].copy()       # 2020-21 -> 2023-24
test_df  = train_df[train_df["SEASON_ID"] == "2023-24"].copy()  # validation season
train_df = train_df[train_df["SEASON_ID"] < "2023-24"].copy()  # earlier seasons for training

stats = ['PTS', 'REB', 'AST', 'STL', 'BLK', 'FG3M', 'FG%', 'FT%', 'TOV']
# target columns: 9 cat z-scores + total
target_cols = [f"next_z_{c}" for c in stats] + ["next_fantasy_z_9cat"]

linear_X_train, y_train = linear_train[feature_cols], train[target_cols]
linear_X_test, y_test = linear_test[feature_cols], test[target_cols]
train_df = pd.concat([linear_X_train, y_train], axis=1)
test_df = pd.concat([linear_X_test, y_test], axis=1)

# train models
ols_models, rf_models, perf_df, feat_names = utils.train_models_per_target(
    train_df=train_df,
    test_df=test_df,
    raw_feature_columns=raw_feature_columns,
    target_cols=target_cols,
    preprocessor=preprocessor,
    rf_params={"n_estimators":300,"max_depth":10,"random_state":42},
)

print("Per-target performance (R2):")
display(perf_df)

# forecast for predict_only (2024-25)
df_forecast = utils.forecast_targets_on_predict_only(
    predict_df=predict_only,
    preprocessor=preprocessor,
    ols_models=ols_models,
    rf_models=rf_models,
    feat_names=feat_names,
    raw_feature_columns=raw_feature_columns,
    target_cols=target_cols
)

# Aggregate predicted total z-score if desired (average over categories)
zcols_ols = [f"pred_OLS_{col}" for col in [f"next_z_{c}" for c in stats]]
zcols_RF = [f"pred_RF_{col}" for col in [f"next_z_{c}" for c in stats]]

df_forecast["pred_OLS_fantasy_z_9cat"] = df_forecast[zcols_ols].mean(axis=1)
df_forecast["pred_RF_fantasy_z_9cat"] = df_forecast[zcols_RF].mean(axis=1)

# inspect top players
df_forecast.sort_values("pred_OLS_fantasy_z_9cat", ascending=False).head(20)


Per-target performance (R2):


,r2_ols,r2_rf
target,,
next_z_PTS,0.802367,0.814476
next_z_REB,0.793509,0.784694
next_z_AST,0.828924,0.800337
next_z_STL,0.478949,0.477391
next_z_BLK,0.795663,0.770997
next_z_FG3M,0.745881,0.739467
next_z_FG%,0.559642,0.588275
next_z_FT%,0.398407,0.587413
next_z_TOV,0.767357,0.750265


,PLAYER_NAME,PLAYER_ID,SEASON_ID,LEAGUE_ID,TEAM_ID,PLAYER_AGE,GP,GS,MIN,FGM,...,pred_OLS_next_z_FG%,pred_RF_next_z_FG%,pred_OLS_next_z_FT%,pred_RF_next_z_FT%,pred_OLS_next_z_TOV,pred_RF_next_z_TOV,pred_OLS_next_fantasy_z_9cat,pred_RF_next_fantasy_z_9cat,pred_OLS_fantasy_z_9cat,pred_RF_fantasy_z_9cat
1921,Victor Wembanyama,NaN,2024-25,NaN,NaN,21.0,46.0,46.0,33.200000,8.900000,...,0.983762,0.702821,1.148311,2.004737,-2.576830,-2.322307,16.719948,10.620498,1.792265,1.516986
1506,Nikola Jokić,203999.0,2024-25,0.0,1.610613e+09,30.0,70.0,70.0,36.728571,11.228571,...,2.233079,3.326438,0.973600,0.876573,-2.545154,-2.580541,15.624058,12.041925,1.649599,1.627024
1740,Shai Gilgeous-Alexander,1628983.0,2024-25,0.0,1.610613e+09,26.0,76.0,76.0,34.184211,11.315789,...,1.468859,1.428538,2.578218,3.601031,-1.917818,-1.262622,14.886174,12.570458,1.625013,1.497953
87,Anthony Davis,203076.0,2024-25,0.0,0.000000e+00,32.0,51.0,51.0,33.431373,9.215686,...,2.579243,2.246697,0.948242,0.383549,-1.367292,-1.039121,12.661495,12.658503,1.376638,1.329787
1312,Luka Dončić,1629029.0,2024-25,0.0,0.000000e+00,26.0,50.0,50.0,35.380000,9.220000,...,0.394819,0.739067,1.150380,0.001529,-2.701808,-2.821878,11.207628,10.524940,1.266261,1.094298
1903,Tyrese Haliburton,1630169.0,2024-25,0.0,1.610613e+09,25.0,73.0,73.0,33.561644,6.520548,...,-0.206838,0.543943,1.664382,0.912781,-1.036018,-0.772975,10.673968,7.151527,1.195288,1.071357
692,Giannis Antetokounmpo,203507.0,2024-25,0.0,1.610613e+09,30.0,67.0,67.0,34.164179,11.835821,...,3.469382,3.199187,0.225335,-2.910196,-2.619711,-2.458760,11.306681,9.195619,1.194360,0.873932
1909,Tyrese Maxey,1630178.0,2024-25,0.0,1.610613e+09,24.0,52.0,52.0,37.692308,9.173077,...,-0.134881,-0.231006,2.129474,2.441521,-1.432728,-1.375207,9.956577,8.011528,1.100820,0.978707
941,Jayson Tatum,1628369.0,2024-25,0.0,1.610613e+09,27.0,72.0,72.0,36.444444,9.194444,...,0.354187,0.513514,1.445872,2.466683,-2.087528,-1.972116,9.810708,8.885444,1.075943,1.309011
898,Jaren Jackson Jr.,1628991.0,2024-25,0.0,1.610613e+09,25.0,74.0,74.0,29.824324,8.000000,...,0.982837,0.800117,1.378236,0.399706,-1.173343,-0.806097,9.140564,9.858244,1.043344,0.942381


In [8]:
#df_forecast.to_csv("forecast_to_2025-26_including_cats.csv", index = False)

## Final conclusions

In this project, I built a complete end-to-end predictive analytics pipeline to forecast NBA player fantasy performance using multi-season historical data, per-category Z-scores, engineered rolling features, and season-over-season deltas. The work included extensive data cleansing, longitudinal feature engineering, creation of per-category next-season targets, and construction of a robust modeling framework designed to outperform a strong baseline (last-season performance → next-season performance).

I trained and compared multiple regression approaches—including OLS with backward elimination, Ridge, Random Forest, XGBoost, and PCA-assisted Linear Regression—and evaluated performance using MAE, RMSE, R², adjusted R², and per-category predictive power. Our best model, OLS with feature selection, improved upon the baseline across all core metrics (MAE 1.750 vs. 1.857 baseline; R² 0.714 vs. 0.669 baseline), while Random Forest achieved the strongest overall category-level R², particularly in AST, REB, BLK, and PTS prediction.
This dual-model setup allows both interpretability (OLS) and performance (RF), and supports downstream recommendation-system logic for fantasy roster construction.

At the target-category level, we found that assists, rebounds, blocks, and points were the most predictable year-over-year, while FG% and FT% remained the most volatile with basketball analytics literature. Feature-importance patterns confirmed that rolling 2-year trends and season-over-season deltas provide substantial predictive lift over raw per-game stats.

Overall, this project demonstrates advanced end-to-end data science capabilities: multi-season time-series feature engineering, creation of domain-specific performance metrics, supervised modelling with feature selection, baseline-beating forecasting, and production-ready prediction logic for future seasons (2024-25). The work integrates statistical modeling, machine learning, and basketball domain expertise to generate actionable insights for fantasy team-building and long-term player value evaluation.